# Migrate to Apache Iceberg
## Dual writes - Full Table Migration of Hive to Iceberg tables

### Configure Spark Session

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
         "executorMemory": "14G",
         "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.sql.catalog.spark_catalog":"org.apache.iceberg.spark.SparkSessionCatalog",
         "spark.sql.catalog.spark_catalog.type":"glue"
    }
}  

### Configure variables

In [ ]:
source_table_name = "events_hive_dual"
target_table_name="events_iceberg_from_hive_dual"
bucket_name="mybucket"
db_name="mydb"

### Create db

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {db_name}
LOCATION 's3://{bucket_name}/{db_name}'
""")

### Create and populate source table

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{source_table_name} (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP,
    additional_data STRING,
    year INT,
    month INT,
    day INT
)
USING PARQUET
PARTITIONED BY (year,month,day)
LOCATION 's3://{bucket_name}/{db_name}/{source_table_name}/'
""")

In [ ]:
#To allow automatic creation of partitions when inserting data
spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{source_table_name}
VALUES 
    ('EVT_001', 'USER_006', 'click', timestamp('2025-10-29 09:15:00'), 'data1', 2025, 10, 29),
    ('EVT_002', 'USER_003', 'purchase', timestamp('2025-10-29 17:30:00'), 'data2', 2025, 10, 29),
    ('EVT_003', 'USER_005', 'view', timestamp('2025-10-30 11:45:00'), 'data3', 2025, 10, 30),
    ('EVT_004', 'USER_001', 'login', timestamp('2025-10-30 14:20:00'), 'data4', 2025, 10, 30),
    ('EVT_005', 'USER_004', 'click', timestamp('2025-10-31 10:10:00'), 'data5', 2025, 10, 31),
    ('EVT_006', 'USER_002', 'purchase', timestamp('2025-10-31 16:30:00'), 'data6', 2025, 10, 31)
""")

Verify Hive table

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{source_table_name} ORDER BY event_id
""").show(truncate=False)

### Step1: Full migration with CTAS

Let's perform full migration to Iceberg. We also implement Iceberg hidden partitioning during the migration.

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{target_table_name}
USING ICEBERG
PARTITIONED BY (days(event_timestamp))
LOCATION 's3://{bucket_name}/{db_name}/{target_table_name}/'
AS (SELECT event_id, user_id, event_type, event_timestamp, additional_data FROM {db_name}.{source_table_name})
""")

Verify Iceberg table

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{target_table_name} ORDER BY event_id
""").show(truncate=False)

### Step 2: Dual Writes

From now on, for every new write operation, write on both the tables

Let's say this is the source data to write

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
from pyspark.sql.functions import to_timestamp

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True), 
    StructField("additional_data", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day", IntegerType(), True)
])

# Create the data
data = [
    ("EVT_007", "USER_006", "click", "2025-11-01 09:15:00", "data7", 2025, 11, 1),
    ("EVT_008", "USER_003", "purchase", "2025-11-01 17:30:00", "data8", 2025, 11, 1),
    ("EVT_009", "USER_005", "view", "2025-11-02 11:45:00", "data9", 2025, 11, 2),
    ("EVT_010", "USER_001", "login", "2025-11-02 14:20:00", "data10", 2025, 11, 2),
    ("EVT_011", "USER_004", "click", "2025-11-03 10:10:00", "data11", 2025, 11, 3),
    ("EVT_012", "USER_002", "purchase", "2025-11-03 16:30:00", "data12", 2025, 11, 3)
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Convert string to timestamp
df = df.withColumn("event_timestamp", to_timestamp("event_timestamp"))

We write to both the tables

In [ ]:
# Insert into Hive table
df.write.insertInto(f"{db_name}.{source_table_name}")

In [ ]:
# Insert into Iceberg table after dropping partition columns
df.drop("year", "month", "day") \
  .write.insertInto(f"{db_name}.{target_table_name}")

Verify tables

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{source_table_name} ORDER BY event_id
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{target_table_name} ORDER BY event_id
""").show(truncate=False)